In [1]:
pip install yfinance pandas

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 22.6 MB/s  0:00:00
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)

   ---------------------------------------- 0/8 [pytz]
   ---------------------------------------- 0/8 [pytz]
   ----- ---------------------------------- 1/8 [peewee]
   --------------- ------------------------ 3/8 [soupsieve]
   ------------------------------ --------- 6/8 [curl_cffi]
   ----------------------------------- ---- 7/8 [yfinance]
   ---------------------------------------- 8/8 [yfinance]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   -------------------------------

In [1]:
import yfinance as yf
import pandas as pd

# Define the tickers
tickers = ["^NSEI", "^BSESN"]

# Fetch data (All columns: Open, High, Low, Close, Volume)
data = yf.download(tickers, period="10y", interval="1d", auto_adjust=True)

# This creates a MultiIndex DataFrame (e.g., ('Open', '^NSEI'), ('Open', '^BSESN'), etc.)
# Let's flatten it for easier use in your ML model
data.columns = ['_'.join(col).strip() for col in data.columns.values]

# Clean up: Drop rows with missing values (weekends/holidays)
data = data.dropna()

print("Columns now available:")
print(data.columns.tolist())

# Save the full dataset
data.to_csv("nifty_sensex_full_10y.csv")

[*********************100%***********************]  2 of 2 completed

Columns now available:
['Close_^BSESN', 'Close_^NSEI', 'High_^BSESN', 'High_^NSEI', 'Low_^BSESN', 'Low_^NSEI', 'Open_^BSESN', 'Open_^NSEI', 'Volume_^BSESN', 'Volume_^NSEI']


In [2]:
vol_zeros_sensex = (data['Volume_^BSESN']== 0).sum()
print(vol_zeros_sensex)

9


In [3]:
vol_zeros_nifty = (data['Volume_^NSEI']== 0).sum()
print(vol_zeros_nifty)

29


In [4]:
data.drop(columns=['Volume_^BSESN','Volume_^NSEI'],inplace = True)


In [25]:
data.head()

,Close_^BSESN,Close_^NSEI,High_^BSESN,High_^NSEI,Low_^BSESN,Low_^NSEI,Open_^BSESN,Open_^NSEI
Date,,,,,,,,
2016-04-28,25603.099609,7847.250000,26100.539062,7992.000000,25561.169922,7834.450195,26078.279297,7967.399902
2016-04-29,25606.619141,7849.799805,25755.429688,7889.049805,25424.029297,7788.700195,25612.910156,7844.250000
2016-05-02,25436.970703,7805.899902,25565.439453,7829.799805,25341.140625,7777.299805,25565.439453,7822.700195
2016-05-03,25229.699219,7747.000000,25705.960938,7890.250000,25192.939453,7735.149902,25500.140625,7824.799805
2016-05-04,25101.730469,7706.549805,25245.699219,7749.000000,25061.039062,7697.250000,25210.869141,7724.149902


In [5]:
# Desired order: Open → High → Low → Close
new_order = [
    'Open_^BSESN', 'Open_^NSEI',
    'Low_^BSESN', 'Low_^NSEI',
    'High_^BSESN', 'High_^NSEI',
    'Close_^BSESN', 'Close_^NSEI'
]

# Reorder columns
data = data[new_order]

print(data.head())

             Open_^BSESN   Open_^NSEI    Low_^BSESN    Low_^NSEI  \
Date                                                               
2016-04-28  26078.279297  7967.399902  25561.169922  7834.450195   
2016-04-29  25612.910156  7844.250000  25424.029297  7788.700195   
2016-05-02  25565.439453  7822.700195  25341.140625  7777.299805   
2016-05-03  25500.140625  7824.799805  25192.939453  7735.149902   
2016-05-04  25210.869141  7724.149902  25061.039062  7697.250000   

             High_^BSESN   High_^NSEI  Close_^BSESN  Close_^NSEI  
Date                                                              
2016-04-28  26100.539062  7992.000000  25603.099609  7847.250000  
2016-04-29  25755.429688  7889.049805  25606.619141  7849.799805  
2016-05-02  25565.439453  7829.799805  25436.970703  7805.899902  
2016-05-03  25705.960938  7890.250000  25229.699219  7747.000000  
2016-05-04  25245.699219  7749.000000  25101.730469  7706.549805  


In [7]:
print(data.shape)

(2461, 8)


In [9]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2461 entries, 2016-04-28 to 2026-04-28
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open_^BSESN   2461 non-null   float64
 1   Open_^NSEI    2461 non-null   float64
 2   Low_^BSESN    2461 non-null   float64
 3   Low_^NSEI     2461 non-null   float64
 4   High_^BSESN   2461 non-null   float64
 5   High_^NSEI    2461 non-null   float64
 6   Close_^BSESN  2461 non-null   float64
 7   Close_^NSEI   2461 non-null   float64
dtypes: float64(8)
memory usage: 173.0 KB


In [10]:
data.describe()

,Open_^BSESN,Open_^NSEI,Low_^BSESN,Low_^NSEI,High_^BSESN,High_^NSEI,Close_^BSESN,Close_^NSEI
count,2461.000000,2461.000000,2461.000000,2461.000000,2461.000000,2461.000000,2461.000000,2461.000000
mean,52017.696601,15682.484076,51705.798773,15589.624131,52258.502875,15753.585246,51982.969540,15673.641865
std,18729.467612,5704.800117,18662.332348,5682.493287,18806.291576,5726.675384,18737.872327,5705.155782
min,25187.660156,7717.649902,25057.929688,7511.100098,25245.699219,7738.899902,25101.730469,7610.250000
25%,35353.960938,10690.549805,35118.421875,10612.849609,35510.011719,10736.150391,35286.738281,10672.250000
50%,50899.578125,15064.400391,50512.839844,14985.849609,51073.269531,15168.250000,50792.078125,15098.400391
75%,66266.351562,19767.000000,65998.898438,19690.199219,66478.898438,19825.550781,66265.562500,19745.000000
max,86065.921875,26333.699219,85577.820312,26210.050781,86159.023438,26373.199219,85836.117188,26328.550781
